**Trade-off Analysis: 7 Abordagens de Agregação de Chunks**

Este notebook compara sete abordagens de processamento de texto para fine-tuning do modelo LegalBERT:
- **Truncation (512)**: Trunca o texto em 512 tokens (baseline)
- **Mean Pooling - Stride 0**: Chunks sem sobreposição + média
- **Mean Pooling - Stride 128**: Chunks com 384 tokens de sobreposição + média
- **Mean Pooling - Stride 256**: Chunks com 256 tokens de sobreposição + média
- **Max Pooling - Stride 0**: Chunks sem sobreposição + máximo
- **Max Pooling - Stride 128**: Chunks com 384 tokens de sobreposição + máximo
- **Max Pooling - Stride 256**: Chunks com 256 tokens de sobreposição + máximo

### Import Libraries and Set the intial variables

In [ ]:
#!pip install transformers

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from pylab import rcParams

from sklearn.metrics import confusion_matrix, classification_report

# Torch ML libraries
from transformers import BertModel, AutoTokenizer, get_linear_schedule_with_warmup
import torch

from torch import nn
from torch.utils.data import Dataset, DataLoader

# Misc.
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Set intial variables and constants
%config InlineBackend.figure_format='retina'

# Graph Designs
sns.set(style='whitegrid', palette='muted', font_scale=1.2)
HAPPY_COLORS_PALETTE = ["#01BEFE", "#FFDD00", "#FF7D00", "#FF006D", "#ADFF02", "#8F00FF"]
sns.set_palette(sns.color_palette(HAPPY_COLORS_PALETTE))
rcParams['figure.figsize'] = 12, 8

# Random seed for reproducibilty
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Set GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


##Parametrização conforme ambiente de execução.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BASE_PATH_GENERAL = '/content/drive/MyDrive/_Mestrado PUCPR/'
BASE_PATH_DATASET = f'{BASE_PATH_GENERAL}Datasets/'

MODEL_CHECKPOINT = 'raquelsilveira/legalbertpt_fp'
NUM_LAYERS = 12

RANDOM_STATE = 42
EPOCHS = 200
APPROACH_EXECUTION = 'epoch' #'step'
LEARNING_RATE = 2e-5
BATCH_SIZE = 8
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
EARLY_STOPPING = 5
class_names = ['Desfavorável ao Réu', 'Favorável ao Réu']

# ---------------------------------------------------------------------------
# Sliding Window — Configurações
# ---------------------------------------------------------------------------
# CHUNK_SIZE: tamanho TOTAL da janela de entrada do modelo (inclui [CLS] e [SEP]).
# O BERT-base suporta no máximo 512 posições.
# Tokens de conteúdo efetivo por chunk = CHUNK_SIZE - 2  →  512 - 2 = 510
CHUNK_SIZE = 512
EFFECTIVE_CHUNK_TOKENS = CHUNK_SIZE - 2  # 510 tokens de conteúdo por chunk (sem [CLS]/[SEP])

STRIDE = 128
MAX_CHUNKS = None
# Desconsidera chunks com menos de MIN_CHUNK_TOKENS tokens reais de conteúdo
# (conta apenas os tokens do texto, sem [CLS], [SEP] e padding)
MIN_CHUNK_TOKENS = 64

# ---------------------------------------------------------------------------
# Abordagens a executar (trade-off Truncation vs Sliding Window)
# ---------------------------------------------------------------------------
# Cada abordagem define explicitamente: label, stride efetivo, max_chunks, batch_size e pooling.
APPROACHES = [
    # ---------------------------------------------------------------------------
    # Abordagens disponíveis — Trade-off Stride vs Qualidade Semântica
    # ---------------------------------------------------------------------------
    #
    # STRIDE 0 (sem sobreposição) → stride efetivo = CHUNK_SIZE
    #   Chunk 1: [ tokens  1 …  510 ]
    #   Chunk 2: [ tokens 511 … 1020 ]
    #   O token 510 e o token 511 NUNCA aparecem juntos na mesma janela de
    #   atenção. Se formavam uma frase ou expressão, o modelo jamais verá esse
    #   contexto combinado → efeito de borda nas fronteiras dos chunks.
    #   Vantagem: menor custo computacional (menos chunks por documento).
    #
    # STRIDE 128 (sobreposição de 382 tokens)
    #   Chunk 1: [ tokens   1 …  510 ]
    #   Chunk 2: [ tokens 129 …  638 ]  ← 382 tokens em comum com chunk 1
    #   Chunk 3: [ tokens 257 …  766 ]  ← 382 tokens em comum com chunk 2
    #   Tokens de fronteira aparecem em múltiplos chunks, preservando o contexto
    #   ao redor deles. Maior cobertura semântica, porém mais chunks por doc.
    #
    # STRIDE 256 (sobreposição de 254 tokens)
    #   Chunk 1: [ tokens   1 …  510 ]
    #   Chunk 2: [ tokens 257 …  766 ]  ← 254 tokens em comum com chunk 1
    #   Chunk 3: [ tokens 513 … 1022 ]  ← 254 tokens em comum com chunk 2
    #   Equilíbrio intermediário entre cobertura semântica e custo computacional.
    #
    # NOTA: "stride=0" é representado internamente como stride=CHUNK_SIZE porque
    #       stride literal 0 causaria loop infinito (janela nunca avançaria).
    #       O label "Stride 0" é semântico: significa "sem sobreposição".
    # ---------------------------------------------------------------------------

    # 1 — Truncation (baseline)
    {
        'label': 'Truncation (512)',
        'stride': CHUNK_SIZE,
        'max_chunks': 1,
        'batch_size': 32,
        'pooling': 'max',
        'min_chunk_tokens': 1,
    },
    # 2 — Mean Pooling, sem sobreposição
    {
        'label': 'Mean Pooling - Stride 0',
        'stride': CHUNK_SIZE,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'mean',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
    },
    # 3 — Mean Pooling, sobreposição de 384 tokens
    {
        'label': 'Mean Pooling - Stride 128',
        'stride': 128,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'mean',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
    },
    # 4 — Mean Pooling, sobreposição de 256 tokens
    {
        'label': 'Mean Pooling - Stride 256',
        'stride': 256,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'mean',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
    },
    # 5 — Max Pooling, sem sobreposição
    {
        'label': 'Max Pooling - Stride 0',
        'stride': CHUNK_SIZE,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'max',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
    },
    # 6 — Max Pooling, sobreposição de 384 tokens
    {
        'label': 'Max Pooling - Stride 128',
        'stride': 128,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'max',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
    },
    # 7 — Max Pooling, sobreposição de 256 tokens
    {
        'label': 'Max Pooling - Stride 256',
        'stride': 256,
        'max_chunks': MAX_CHUNKS,
        'batch_size': BATCH_SIZE,
        'pooling': 'max',
        'min_chunk_tokens': MIN_CHUNK_TOKENS,
    },
]

# Performance knobs
NUM_WORKERS = 2

In [ ]:
import os

# Paths padrão para reprodutibilidade
train_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2026_train-punct.csv"
test_path = f"{BASE_PATH_DATASET}preprocessed_acordaos_HC_under_2021_2026_test-punct.csv"

# Apenas carrega os arquivos já persistidos (sem split)
if not os.path.exists(train_path):
    raise FileNotFoundError(f"Arquivo de treino não encontrado: {train_path}")
if not os.path.exists(test_path):
    raise FileNotFoundError(f"Arquivo de teste não encontrado: {test_path}")

print("Carregando arquivos de treino e teste persistidos...")
df_train = pd.read_csv(train_path, sep=';')
df_test = pd.read_csv(test_path, sep=';')

print(f"Treino carregado de: {train_path} | shape: {df_train.shape}")
print(f"Teste carregado de: {test_path} | shape: {df_test.shape}")

In [ ]:
# Function para converter a decisão em valor inteiro (0 = Desfavorável, 1 = Favorável)
def cast_decisao_to_int(decisao_favoravel):
    if decisao_favoravel == 'S':
        return 1
    else:
        return 0

# Apply to train and test datasets
if 'DECISAO_FAVORAVEL' in df_train.columns and 'classe' not in df_train.columns:
    df_train['classe'] = df_train.DECISAO_FAVORAVEL.apply(cast_decisao_to_int)
if 'DECISAO_FAVORAVEL' in df_test.columns and 'classe' not in df_test.columns:
    df_test['classe'] = df_test.DECISAO_FAVORAVEL.apply(cast_decisao_to_int)

print(f"Classes no treino: {df_train['classe'].value_counts().to_dict()}")
print(f"Classes no teste:  {df_test['classe'].value_counts().to_dict()}")

# BERT

### Preparing Torch Dataset

###  `HCAcordaoSTJ` (Custom Dataset com Sliding Window)

In [ ]:
from typing import Dict, List, Union
import numpy as np
import torch
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence

class HCAcordaoSTJ(Dataset):
    """Dataset com Sliding Window para textos longos."""

    def __init__(self, reports: np.ndarray, targets: np.ndarray, tokenizer,
                 chunk_size: int, stride: int, max_chunks: int = None,
                 min_chunk_tokens: int = 1) -> None:
        self.reports = reports.astype(str)
        self.targets = targets
        self.chunk_size = chunk_size
        self.stride = stride
        self.max_chunks = max_chunks
        self.min_chunk_tokens = max(1, min_chunk_tokens)

        # Tokens especiais [CLS] (id=101) e [SEP] (id=102) ocupam 2 posições.
        # effective_length = tokens de CONTEÚDO por chunk = chunk_size - 2
        # Ex.: chunk_size=512  →  effective_length=510 tokens de conteúdo
        #      input final:  [CLS] + 510 tokens + [SEP]  =  512  (limite do modelo)
        self.effective_length = chunk_size - 2

        # Tokeniza sem truncar para obter todos os tokens
        all_encodings = tokenizer(
            self.reports.tolist(),
            add_special_tokens=False,
            return_attention_mask=False,
        )

        # Cria chunks por documento
        self.all_input_ids: List[torch.Tensor] = []
        self.all_attention_masks: List[torch.Tensor] = []

        cls_id = tokenizer.cls_token_id
        sep_id = tokenizer.sep_token_id
        pad_id = tokenizer.pad_token_id if tokenizer.pad_token_id is not None else 0

        # Contadores para relatório de impacto do filtro min_chunk_tokens
        total_chunks_before_filter = 0
        total_chunks_after_filter = 0
        total_chunks_after_max = 0

        for doc_tokens in all_encodings['input_ids']:
            chunks_ids = []
            chunks_mask = []
            doc_chunks_before = 0

            # Gera janelas deslizantes
            start = 0
            while start < len(doc_tokens):
                end = start + self.effective_length
                window_tokens = doc_tokens[start:end]

                doc_chunks_before += 1

                # Descarta chunks muito curtos quando ja existem chunks validos.
                # Fallback: se o documento for muito curto, mantemos ao menos 1 chunk.
                if len(window_tokens) < self.min_chunk_tokens and len(chunks_ids) > 0:
                    if end >= len(doc_tokens):
                        break
                    start += self.stride
                    continue

                # Adiciona [CLS] ... tokens ... [SEP]
                input_ids = [cls_id] + window_tokens + [sep_id]

                # Padding até chunk_size
                pad_len = chunk_size - len(input_ids)
                attention_mask = [1] * len(input_ids) + [0] * pad_len
                input_ids = input_ids + [pad_id] * pad_len

                chunks_ids.append(input_ids)
                chunks_mask.append(attention_mask)

                # Se já cobriu todo o documento, para
                if end >= len(doc_tokens):
                    break
                start += self.stride

            total_chunks_before_filter += doc_chunks_before
            total_chunks_after_filter += len(chunks_ids)

            # Limita número de chunks se configurado
            if self.max_chunks is not None:
                chunks_ids = chunks_ids[:self.max_chunks]
                chunks_mask = chunks_mask[:self.max_chunks]

            total_chunks_after_max += len(chunks_ids)

            # Garante ao menos um chunk por documento
            if len(chunks_ids) == 0:
                window_tokens = doc_tokens[:self.effective_length]
                input_ids = [cls_id] + window_tokens + [sep_id]
                pad_len = chunk_size - len(input_ids)
                attention_mask = [1] * len(input_ids) + [0] * pad_len
                input_ids = input_ids + [pad_id] * pad_len
                chunks_ids.append(input_ids)
                chunks_mask.append(attention_mask)

            self.all_input_ids.append(torch.tensor(chunks_ids, dtype=torch.long))
            self.all_attention_masks.append(torch.tensor(chunks_mask, dtype=torch.long))

        # --- Relatório de impacto do filtro min_chunk_tokens ---
        n_docs = len(self.reports)
        avg_before = total_chunks_before_filter / n_docs if n_docs > 0 else 0.0
        avg_after_filter = total_chunks_after_filter / n_docs if n_docs > 0 else 0.0
        avg_after_max = total_chunks_after_max / n_docs if n_docs > 0 else 0.0
        removed = total_chunks_before_filter - total_chunks_after_filter
        pct_removed = 100.0 * removed / total_chunks_before_filter if total_chunks_before_filter > 0 else 0.0

        print(f"  [Dataset] Documentos: {n_docs}")
        print(f"  [Dataset] Chunks/doc antes do filtro (min={self.min_chunk_tokens}):  {avg_before:.2f}  (total={total_chunks_before_filter})")
        print(f"  [Dataset] Chunks/doc após  filtro (min={self.min_chunk_tokens}):  {avg_after_filter:.2f}  (removidos={removed}, {pct_removed:.1f}%)")
        if self.max_chunks is not None:
            print(f"  [Dataset] Chunks/doc após max_chunks={self.max_chunks}:  {avg_after_max:.2f}  (total={total_chunks_after_max})")
        else:
            print(f"  [Dataset] max_chunks=None — sem limitação adicional")

    def __len__(self) -> int:
        return len(self.reports)

    def __getitem__(self, item: int) -> Dict[str, Union[str, torch.Tensor]]:
        return {
            'report_text': self.reports[item],
            'input_ids': self.all_input_ids[item],
            'attention_mask': self.all_attention_masks[item],
            'num_chunks': self.all_input_ids[item].shape[0],
            'targets': torch.tensor(self.targets[item], dtype=torch.long)
        }


def sliding_window_collate_fn(batch: List[Dict]) -> Dict[str, Union[torch.Tensor, List[str]]]:
    """Collate function para batches com número variável de chunks por documento."""
    all_ids, all_masks, targets, texts = [], [], [], []
    chunk_counts = []

    for sample in batch:
        all_ids.append(sample['input_ids'])
        all_masks.append(sample['attention_mask'])
        targets.append(sample['targets'])
        texts.append(sample['report_text'])
        chunk_counts.append(sample['num_chunks'])

    # Concatena todos os chunks num tensor (total_chunks, chunk_size)
    input_ids = torch.cat(all_ids, dim=0)
    attention_mask = torch.cat(all_masks, dim=0)
    targets = torch.stack(targets)
    chunk_counts = torch.tensor(chunk_counts, dtype=torch.long)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'targets': targets,
        'chunk_counts': chunk_counts,
        'report_text': texts
    }

In [ ]:
if 'df_train' not in globals() or 'df_test' not in globals():
    raise RuntimeError("Execute primeiro a célula de carga inteligente para criar/carregar treino e teste.")

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

In [ ]:
import os

def create_data_loader(df: pd.DataFrame, tokenizer, chunk_size: int, stride: int,
                       batch_size: int, max_chunks: int = None,
                       min_chunk_tokens: int = 1, shuffle: bool = False) -> DataLoader:
    ds = HCAcordaoSTJ(
        reports=df.RELATORIO.to_numpy(),
        targets=df.classe.to_numpy(),
        tokenizer=tokenizer,
        chunk_size=chunk_size,
        stride=stride,
        max_chunks=max_chunks,
        min_chunk_tokens=min_chunk_tokens
    )

    workers = min(NUM_WORKERS, os.cpu_count() or 1)

    loader_kwargs = {
        'dataset': ds,
        'batch_size': batch_size,
        'shuffle': shuffle,
        'num_workers': workers,
        'pin_memory': (device.type == 'cuda'),
        'collate_fn': sliding_window_collate_fn
    }

    if workers > 0:
        loader_kwargs['persistent_workers'] = True
        loader_kwargs['prefetch_factor'] = 2

    return DataLoader(**loader_kwargs)

In [ ]:
class HCAcordaoClassifier(nn.Module):

    # Mapa de arquitetura da cabeça classificadora por número de camadas do modelo
    HEAD_CONFIG = {
        12: (5_000, 2_500, 1_250, 750),
    }

    def __init__(self, n_classes: int, num_layers: int, model_name: str, pooling_strategy: str = 'max') -> None:
        super(HCAcordaoClassifier, self).__init__()
        self.bert = BertModel.from_pretrained(model_name)
        self.pooling_strategy = pooling_strategy

        if self.pooling_strategy not in {'max', 'mean'}:
            raise ValueError(f"pooling_strategy invalida ({self.pooling_strategy}). Use 'max' ou 'mean'.")

        if num_layers not in self.HEAD_CONFIG:
            raise ValueError(f"num_layers inválido ({num_layers}). Valores suportados: {list(self.HEAD_CONFIG.keys())}")

        h1, h2, h3, h4 = self.HEAD_CONFIG[num_layers]

        self.fc1 = nn.Linear(self.bert.config.hidden_size, h1)
        self.drop1 = nn.Dropout(p=0.3)
        self.relu1 = nn.ReLU()

        self.fc2 = nn.Linear(h1, h2)
        self.drop2 = nn.Dropout(p=0.3)
        self.relu2 = nn.ReLU()

        self.fc3 = nn.Linear(h2, h3)
        self.drop3 = nn.Dropout(p=0.3)
        self.relu3 = nn.ReLU()

        self.fc4 = nn.Linear(h3, h4)
        self.drop4 = nn.Dropout(p=0.3)
        self.relu4 = nn.ReLU()

        self.out = nn.Linear(h4, n_classes)

        self.freeze_layers(num_layers)

    def freeze_layers(self, num_layers_to_freeze: int) -> None:
        print(f"Freezing embeddings + first {num_layers_to_freeze} encoder layers...")
        for name, param in self.bert.named_parameters():
            if 'embeddings' in name:
                param.requires_grad = False
            elif 'encoder.layer' in name:
                layer_num = int(name.split('.')[2])
                if layer_num < num_layers_to_freeze:
                    param.requires_grad = False
                else:
                    param.requires_grad = True
            else:
                param.requires_grad = True
        print("Layers freezing complete.")

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor,
                chunk_counts: torch.Tensor = None) -> torch.Tensor:
        """Forward pass com suporte a Sliding Window."""
        output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        chunk_embeddings = output.pooler_output

        # Agrega chunks por documento se sliding window está ativo
        if chunk_counts is not None and chunk_counts.sum() != chunk_counts.shape[0]:
            # Agrupa os embeddings por documento e aplica a estratégia de pooling
            doc_embeddings = []
            idx = 0
            for count in chunk_counts:
                c = count.item()
                doc_chunks = chunk_embeddings[idx:idx + c]
                if self.pooling_strategy == 'max':
                    pooled, _ = doc_chunks.max(dim=0)
                else:
                    pooled = doc_chunks.mean(dim=0)
                doc_embeddings.append(pooled)
                idx += c
            hidden = torch.stack(doc_embeddings)
        else:
            hidden = chunk_embeddings

        hidden = self.fc1(hidden)
        hidden = self.relu1(hidden)
        hidden = self.drop1(hidden)

        hidden = self.fc2(hidden)
        hidden = self.relu2(hidden)
        hidden = self.drop2(hidden)

        hidden = self.fc3(hidden)
        hidden = self.relu3(hidden)
        hidden = self.drop3(hidden)

        hidden = self.fc4(hidden)
        hidden = self.relu4(hidden)
        hidden = self.drop4(hidden)

        return self.out(hidden)


In [ ]:
%%time
import os, gc, copy
from collections import defaultdict
from torch.optim import AdamW
from sklearn.metrics import precision_score, recall_score, f1_score
import torch.nn.functional as F
from sklearn.metrics import roc_curve, auc

# ---------------------------------------------------------------------------
# Funções auxiliares
# ---------------------------------------------------------------------------
def train_epoch(model, data_loader, loss_fn, optimizer, device, scheduler, n_examples, scaler, use_amp):
    model.train()
    losses, correct_predictions = [], 0
    all_preds, all_targets = [], []
    optimizer.zero_grad(set_to_none=True)

    for d in data_loader:
        input_ids = d["input_ids"].to(device, non_blocking=True)
        attention_mask = d["attention_mask"].to(device, non_blocking=True)
        targets = d["targets"].to(device, non_blocking=True)
        chunk_counts = d["chunk_counts"].to(device, non_blocking=True)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, chunk_counts=chunk_counts)
            loss = loss_fn(outputs, targets)

        _, preds = torch.max(outputs, dim=1)
        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())
        all_preds.extend(preds.detach().cpu().numpy())
        all_targets.extend(targets.detach().cpu().numpy())

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)

    acc = (correct_predictions.double() / n_examples).item()
    return (acc, np.mean(losses),
            precision_score(all_targets, all_preds, average='macro', zero_division=0),
            recall_score(all_targets, all_preds, average='macro', zero_division=0),
            f1_score(all_targets, all_preds, average='macro', zero_division=0))


def eval_model(model, data_loader, loss_fn, device, n_examples, use_amp=False):
    model.eval()
    losses, correct_predictions = [], 0
    all_preds, all_targets = [], []

    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device, non_blocking=True)
            attention_mask = d["attention_mask"].to(device, non_blocking=True)
            targets = d["targets"].to(device, non_blocking=True)
            chunk_counts = d["chunk_counts"].to(device, non_blocking=True)

            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, chunk_counts=chunk_counts)
                loss = loss_fn(outputs, targets)

            _, preds = torch.max(outputs, dim=1)
            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(targets.cpu().numpy())

    acc = (correct_predictions.double() / n_examples).item()
    return (acc, np.mean(losses),
            precision_score(all_targets, all_preds, average='macro', zero_division=0),
            recall_score(all_targets, all_preds, average='macro', zero_division=0),
            f1_score(all_targets, all_preds, average='macro', zero_division=0))


def get_predictions(model, data_loader, device):
    model.eval()
    predictions, prediction_probs, real_values = [], [], []
    with torch.no_grad():
        for d in data_loader:
            input_ids = d["input_ids"].to(device, non_blocking=True)
            attention_mask = d["attention_mask"].to(device, non_blocking=True)
            targets = d["targets"].to(device, non_blocking=True)
            chunk_counts = d["chunk_counts"].to(device, non_blocking=True)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, chunk_counts=chunk_counts)
            _, preds = torch.max(outputs, dim=1)
            predictions.extend(preds)
            prediction_probs.extend(outputs)
            real_values.extend(targets)
    return (torch.stack(predictions).cpu(),
            torch.stack(prediction_probs).cpu(),
            torch.stack(real_values).cpu())


# ---------------------------------------------------------------------------
# Loop de treinamento — Trade-off: Truncation vs Sliding Window
# ---------------------------------------------------------------------------
model_dir = f'{BASE_PATH_GENERAL}models'
os.makedirs(model_dir, exist_ok=True)

# Armazena resultados para comparação final
all_roc_data = []   # (label, fpr, tpr, roc_auc)
all_metrics = []    # dict por abordagem

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

for approach_idx, approach_cfg in enumerate(APPROACHES):
    APPROACH_LABEL = approach_cfg['label']
    APPROACH_STRIDE = approach_cfg['stride']
    APPROACH_MAX_CHUNKS = approach_cfg['max_chunks']
    APPROACH_BATCH_SIZE = approach_cfg['batch_size']
    APPROACH_POOLING = approach_cfg['pooling']
    APPROACH_MIN_CHUNK_TOKENS = approach_cfg.get('min_chunk_tokens', MIN_CHUNK_TOKENS)

    print("\n" + "=" * 80)
    print(f"[{approach_idx+1}/{len(APPROACHES)}] [{approach_idx+1}] {APPROACH_LABEL}")
    print(
        f"  Modelo: {MODEL_CHECKPOINT} | chunk_size={CHUNK_SIZE} | stride={APPROACH_STRIDE} "
        f"| min_chunk_tokens={APPROACH_MIN_CHUNK_TOKENS} | max_chunks={APPROACH_MAX_CHUNKS} | pooling={APPROACH_POOLING}"
    )
    print("=" * 80)

    # --- Reproducibilidade ---
    np.random.seed(RANDOM_SEED)
    torch.manual_seed(RANDOM_SEED)

    # --- DataLoaders ---
    train_data_loader = create_data_loader(
        df_train,
        tokenizer,
        CHUNK_SIZE,
        APPROACH_STRIDE,
        APPROACH_BATCH_SIZE,
        APPROACH_MAX_CHUNKS,
        APPROACH_MIN_CHUNK_TOKENS,
        shuffle=True,
    )
    test_data_loader = create_data_loader(
        df_test,
        tokenizer,
        CHUNK_SIZE,
        APPROACH_STRIDE,
        APPROACH_BATCH_SIZE,
        APPROACH_MAX_CHUNKS,
        APPROACH_MIN_CHUNK_TOKENS,
        shuffle=False,
    )

    # --- Modelo ---
    model = HCAcordaoClassifier(len(class_names), NUM_LAYERS, MODEL_CHECKPOINT, pooling_strategy=APPROACH_POOLING)
    model = model.to(device)

    # --- Optimizer / Scheduler / Loss ---
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    total_steps = len(train_data_loader) * EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
    loss_fn = nn.CrossEntropyLoss().to(device)
    use_amp = (device.type == 'cuda')
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

    # --- Treinamento ---
    history = defaultdict(list)
    best_recall, best_model_weights = 0, None
    epochs_no_improve = 0

    for epoch in range(EPOCHS):
        print(f"\n  Epoch {epoch+1}/{EPOCHS}")
        print("  " + "-" * 60)

        train_acc, train_loss, train_prec, train_rec, train_f1 = train_epoch(
            model, train_data_loader, loss_fn, optimizer, device, scheduler, len(df_train), scaler, use_amp)
        print(f"  Train | loss {train_loss:.4f} acc {train_acc:.4f} prec {train_prec:.4f} rec {train_rec:.4f} f1 {train_f1:.4f}")

        test_acc, test_loss, test_prec, test_rec, test_f1 = eval_model(
            model, test_data_loader, loss_fn, device, len(df_test), use_amp)
        print(f"  Test  | loss {test_loss:.4f} acc {test_acc:.4f} prec {test_prec:.4f} rec {test_rec:.4f} f1 {test_f1:.4f}")

        history['train_loss'].append(train_loss)
        history['test_loss'].append(test_loss)

        if test_rec > best_recall:
            best_model_weights = copy.deepcopy(model.state_dict())
            best_recall = test_rec
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= EARLY_STOPPING:
                print(f"  Early Stopping na época {epoch+1}.")
                break

    # --- Carregar melhores pesos ---
    if best_model_weights is not None:
        model.load_state_dict(best_model_weights)
        safe_label = APPROACH_LABEL.replace(" ", "_").replace("(", "").replace(")", "").replace(",", "").replace("=", "")
        model_save_path = f'{model_dir}/best_model_{MODEL_CHECKPOINT.replace("/","_")}_{safe_label}.bin'
        torch.save(best_model_weights, model_save_path)
        print(f"  Melhor modelo salvo em: {model_save_path}")

    # =======================================================================
    # Resultados individuais — Gráfico de Loss
    # =======================================================================
    plt.figure()
    plt.plot(history['train_loss'], label='train loss')
    plt.plot(history['test_loss'], label='val loss')
    plt.title(f'[{approach_idx+1}] Training vs Validation Loss — {APPROACH_LABEL}')
    plt.ylabel('Loss'); plt.xlabel('Epoch'); plt.legend(); plt.ylim([0, 2])
    plt.show()

    # =======================================================================
    # Resultados individuais — Métricas Finais
    # =======================================================================
    test_acc, test_loss, test_prec, test_rec, test_f1 = eval_model(
        model, test_data_loader, loss_fn, device, len(df_test), use_amp)

    print(f"\n  {'='*60}")
    print(f"  Métricas Finais — [{approach_idx+1}] {APPROACH_LABEL}")
    print(f"  {'='*60}")
    print(f"  Accuracy:  {test_acc:.4f}")
    print(f"  Precision: {test_prec:.4f}")
    print(f"  Recall:    {test_rec:.4f}")
    print(f"  F1-Score:  {test_f1:.4f}")

    # =======================================================================
    # Resultados individuais — Classification Report
    # =======================================================================
    y_pred, y_pred_probs, y_test = get_predictions(model, test_data_loader, device)
    print(f"\n  Classification Report — [{approach_idx+1}] {APPROACH_LABEL}")
    print(classification_report(y_test, y_pred, target_names=class_names))

    # =======================================================================
    # Resultados individuais — Matriz de Confusão
    # =======================================================================
    cm = confusion_matrix(y_test, y_pred)
    df_cm = pd.DataFrame(cm, index=class_names, columns=class_names)

    # FPR no ponto de decisão atual (matriz de confusão)
    if cm.shape == (2, 2):
        tn, fp, fn, tp = cm.ravel()
        fpr_metric = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    else:
        fpr_metric = np.nan

    plt.figure()
    hmap = sns.heatmap(df_cm, annot=True, fmt="d", cmap="Blues")
    hmap.yaxis.set_ticklabels(hmap.yaxis.get_ticklabels(), rotation=0, ha='right')
    hmap.xaxis.set_ticklabels(hmap.xaxis.get_ticklabels(), rotation=30, ha='right')
    plt.ylabel('Resultado do Julgamento')
    plt.xlabel('Decisão Predita')
    plt.title(f'[{approach_idx+1}] Matriz de Confusão — {APPROACH_LABEL}')
    plt.show()
    print(f"  FPR: {fpr_metric:.4f}" if not np.isnan(fpr_metric) else "  FPR: n/a")

    # =======================================================================
    # Resultados individuais — Curva ROC
    # =======================================================================
    probs = F.softmax(y_pred_probs, dim=1).numpy()
    y_scores = probs[:, 1]
    fpr, tpr, _ = roc_curve(y_test.numpy(), y_scores)
    roc_auc = auc(fpr, tpr)

    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
    plt.plot([0, 1], [0, 1], 'navy', lw=2, linestyle='--')
    plt.xlim([0, 1]); plt.ylim([0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title(f'[{approach_idx+1}] Curva ROC — {APPROACH_LABEL}')
    plt.legend(loc="lower right")
    plt.grid(alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.show()
    print(f"  AUC: {roc_auc:.4f}")

    # =======================================================================
    # Resultados individuais — Armazena para comparação final
    # =======================================================================
    all_roc_data.append((approach_idx + 1, APPROACH_LABEL, fpr, tpr, roc_auc))
    all_metrics.append({
        'ID': approach_idx + 1,
        'Model': MODEL_CHECKPOINT,
        'Approach': APPROACH_LABEL,
        'Accuracy': round(test_acc, 4),
        'Precision': round(test_prec, 4),
        'Recall': round(test_rec, 4),
        'FPR': round(float(fpr_metric), 4) if not np.isnan(fpr_metric) else np.nan,
        'F1-Score': round(test_f1, 4),
        'AUC': round(roc_auc, 4),
    })

    # --- Liberar memória ---
    del model, optimizer, scheduler, scaler
    del train_data_loader, test_data_loader
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

print("\n\n" + "=" * 80)
print("Trade-off concluído — 7 abordagens (Truncation | Mean/Max Pooling × Stride 0/128/256).")
print("=" * 80)

## Comparação Final — Curva ROC

In [ ]:
plt.figure(figsize=(10, 8))
colors = ['#FF006D', '#01BEFE', '#FFDD00', '#8F00FF', '#ADFF02', '#FF7D00']

for i, (approach_id, label, fpr, tpr, roc_auc) in enumerate(all_roc_data):
    plt.plot(fpr, tpr, color=colors[i % len(colors)], lw=2.5,
             label=f'[{approach_id}] {label} (AUC = {roc_auc:.4f})')

plt.plot([0, 1], [0, 1], 'navy', lw=2, linestyle='--', label='Random')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate', fontsize=12)
plt.ylabel('True Positive Rate', fontsize=12)
plt.title('Curva ROC Comparativa — 7 Abordagens', fontsize=14)
plt.legend(loc="lower right", fontsize=11)
plt.grid(alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()

## Tabela Comparativa de Metricas

In [ ]:
df_metrics = pd.DataFrame(all_metrics)
df_metrics = df_metrics.sort_values('ID', ascending=True).reset_index(drop=True)

print("=" * 80)
print("TABELA COMPARATIVA — 7 ABORDAGENS (Truncation | Mean/Max Pooling × Stride 0/128/256)")
print("=" * 80)
display(
    df_metrics.style
    .highlight_max(subset=['Accuracy', 'Precision', 'Recall', 'F1-Score', 'AUC'], color='lightgreen')
    .highlight_min(subset=['FPR'], color='lightblue')
)
